# 1. Import Libraries

In [1]:
import numpy as np
import tensorflow as tf
import itertools
import sys
import os
import gc
import json

# 2. Data & Model Paths

## 1. Folders

In [2]:
scripts_folder = os.path.abspath(os.path.join('..', 'Scripts'))
data_folder = os.path.abspath(os.path.join('..', 'Data', 'TrainTest'))
hyper_parameter_folder = os.path.abspath(os.path.join('..', 'Data', 'HyperParameters'))

## 2. System paths

In [3]:
sys.path.append(scripts_folder)

# 3. Import Data & Models

## 1. Import Models

In [4]:
from model import Encoder, Decoder, VAE, RSVD

## 2. Import Data

In [5]:
train_data_path = os.path.join(data_folder, 'train_data.npy')

try:
    train_data = np.load(train_data_path).astype(np.float32)
    train_data_tf = tf.constant(train_data, dtype=tf.float32) # Mencegah tensorflow duplikasi data tiap iterasi (memory leak)
    num_items = train_data.shape[1]
    print(f"Data latih berhasil dimuat.")
    print(f"   Dimensi data: {train_data.shape} (Pengguna x Film)")
except FileNotFoundError:
    print("Error: File 'train_data.npy' tidak ditemukan. Pastikan Anda sudah menjalankan preprocessing.")

Data latih berhasil dimuat.
   Dimensi data: (943, 1682) (Pengguna x Film)


# 4. Hyperparameter Tuning

## 1.VAE

### 1. Hyperparameter Search Space

In [ ]:
vae_param_grid = {
    'latent_dim': [10, 20, 50, 100],         # Sangat padat hingga sangat detail
    'learning_rate': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],   # Rentang kecepatan belajar yang sangat luas
    'batch_size': [64, 128, 256],             # Ukuran batch
    'dropout_rate': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6],    # Mencari titik keseimbangan overfitting
    'hidden_dims': [
        [512, 256],               # Standar
        [256, 128],               # Dangkal
        [512, 256, 128],          # Dalam
    ],
    'beta': [0.1, 0.2, 0.5, 1.0]
}

# Membuat semua kemungkinan kombinasi
vae_keys, vae_values = zip(*vae_param_grid.items())
vae_combinations = [dict(zip(vae_keys, v)) for v in itertools.product(*vae_values)]

best_vae_loss = float('inf')
best_vae_params = None
best_vae_model = None

### 2. Grid Search

#### 1. Load Progress

In [7]:
best_vae_path = os.path.join(hyper_parameter_folder, 'best_vae_weights.weights.h5')
vae_progress_file = os.path.join(hyper_parameter_folder, 'tuning_progress_vae.json')

if os.path.exists(vae_progress_file):
    with open(vae_progress_file, 'r') as f:
        progress_data = json.load(f)
    
    best_vae_loss = progress_data['best_loss']
    best_vae_params = progress_data['best_params']
    completed_indices = progress_data['completed_indices']
    
    print(f"\n[INFO] Melanjutkan sesi sebelumnya...")
    print(f"[INFO] {len(completed_indices)} kombinasi sudah dievaluasi.")
    print(f"[INFO] Loss terbaik saat ini: {best_vae_loss:.4f}")
else:
    best_vae_loss = float('inf')
    best_vae_params = None
    completed_indices = []


[INFO] Melanjutkan sesi sebelumnya...
[INFO] 3600 kombinasi sudah dievaluasi.
[INFO] Loss terbaik saat ini: 54.7315


#### 2. Looping Function

In [ ]:
def train_and_evaluate_vae(params, data, current_best_loss, save_path):
    # Selalu bersihkan sesi di awal sebelum membuat model baru
    tf.keras.backend.clear_session()
    tf.compat.v1.reset_default_graph()
    
    encoder = Encoder(hidden_dims=params['hidden_dims'], latent_dim=params['latent_dim'], dropout_rate=params['dropout_rate'])
    decoder = Decoder(hidden_dims=params['hidden_dims'][::-1], output_dim=num_items)
    vae = VAE(encoder, decoder, beta=params['beta'])
    
    # Gunakan nama variabel spesifik (jangan '_') agar tidak ditahan oleh Jupyter
    dummy_output = vae(data[:1]) 
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=params['learning_rate'])
    vae.compile(optimizer=optimizer, run_eagerly=True) # Model di run per baris secara dinamis
    
    # Melatih model
    history = vae.fit(data, data, epochs=15, batch_size=params['batch_size'], verbose=0)
    final_loss = history.history['loss'][-1]
    
    is_new_best = False
    # Evaluasi dan simpan bobot langsung di dalam fungsi saat model masih "hidup"
    if final_loss < current_best_loss:
        is_new_best = True
        vae.save_weights(save_path)
        
    # Hapus semua variabel lokal sebelum keluar dari fungsi
    del encoder, decoder, vae, history, optimizer, dummy_output
    
    # Kembalikan hanya angka (float) dan boolean, bukan objek model
    return final_loss, is_new_best

#### 3. Looping

In [9]:
for i, params in enumerate(vae_combinations):
    if i in completed_indices:
        continue
    
    print(f"\n[{i+1}/{len(vae_combinations)}] Menguji VAE | Params: {params}")
    
    # Panggil fungsi terisolasi (semua memori model akan hancur otomatis setelah baris ini selesai)
    final_loss, is_new_best = train_and_evaluate_vae(
        params=params, 
        data=train_data_tf, 
        current_best_loss=best_vae_loss, 
        save_path=best_vae_path
    )
    
    print(f"   -> Final Total Loss: {final_loss:.4f}")
    
    # Update tracking jika mendapat hasil terbaik
    if is_new_best:
        print(f"   🌟 [NEW BEST FOUND!] Loss turun ke {final_loss:.4f}. Model telah tersimpan.")
        best_vae_loss = final_loss
        best_vae_params = params

    # Simpan progress JSON
    completed_indices.append(i)
    with open(vae_progress_file, 'w') as f:
        json.dump({
            'best_loss': best_vae_loss,
            'best_params': best_vae_params,
            'completed_indices': completed_indices
        }, f)

    # Sapu bersih RAM setelah fungsi tertutup
    gc.collect()
    tf.keras.backend.clear_session()

print("\n==================================================")
print(f"[PENCARIAN SELESAI] Hasil Terbaik VAE:")
print(f"Parameter: {best_vae_params}")
print(f"Loss Terendah: {best_vae_loss:.4f}")
print(f"Bobot tersimpan di folder: {best_vae_path}")
print("==================================================")


[PENCARIAN SELESAI] Hasil Terbaik VAE:
Parameter: {'latent_dim': 10, 'learning_rate': 0.005, 'batch_size': 64, 'dropout_rate': 0.5, 'hidden_dims': [256, 128]}
Loss Terendah: 54.7315
Bobot tersimpan di folder: c:\Users\Basudewa\Documents\Mata Kuliah\Ujian Proposal\Programs\Data\HyperParameters\best_vae_weights.weights.h5


### 3. Construct Best VAE Model

#### 1. Read Progress Files

In [10]:
try:
    with open(vae_progress_file, 'r') as f:
        best_vae_params = json.load(f)['best_params']
    print(f"[INFO] Blueprint parameter terbaik ditemukan: {best_vae_params}")
except FileNotFoundError:
    print("[ERROR] File tuning_progress_vae.json tidak ditemukan.")

[INFO] Blueprint parameter terbaik ditemukan: {'latent_dim': 10, 'learning_rate': 0.005, 'batch_size': 64, 'dropout_rate': 0.5, 'hidden_dims': [256, 128]}


#### 2. Construct Model

In [11]:
best_encoder = Encoder(
    hidden_dims=best_vae_params['hidden_dims'], 
    latent_dim=best_vae_params['latent_dim'], 
    dropout_rate=best_vae_params['dropout_rate']
)

best_decoder = Decoder(
    hidden_dims=best_vae_params['hidden_dims'][::-1], 
    output_dim=num_items
)

best_vae = VAE(best_encoder, best_decoder)

#### 3. Insert Weights

In [12]:
_ = best_vae(train_data[:1])
best_vae.load_weights(best_vae_path)

### 4. Extract Latent Representation

In [13]:
best_batch = best_vae_params['batch_size']
z_mean, _ = best_vae.encoder.predict(train_data, batch_size=best_batch)

latent_matrix_Z = z_mean
print(latent_matrix_Z.shape)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
(943, 10)


## 2. RSVD

### 1. Hyperparameter Search Space

In [14]:
rsvd_param_grid = {
    'n_factors': [10, 20, 50],              
    # Pertahankan learning rate di kecepatan menengah ke lambat
    'learning_rate': [0.001, 0.005, 0.01, 0.02], 
    # Geser grid lambda ke area emas yang kamu temukan
    'lambda_reg': [0.001, 0.005, 0.01, 0.02]      
}

rsvd_keys, rsvd_values = zip(*rsvd_param_grid.items())
rsvd_combinations = [dict(zip(rsvd_keys, v)) for v in itertools.product(*rsvd_values)]

best_rsvd_error = float('inf')
best_rsvd_params = None
best_rsvd_model = None

### 2. Grid Search

#### 1. Load Progress

In [15]:
rsvd_progress_file = os.path.join(hyper_parameter_folder, 'tuning_progress_rsvd.json')

if os.path.exists(rsvd_progress_file):
    with open(rsvd_progress_file, 'r') as f:
        progress_data = json.load(f)
    
    best_rsvd_error = progress_data['best_error']
    best_rsvd_params = progress_data['best_params']
    completed_indices = progress_data['completed_indices']
    
    print(f"\n[INFO] Melanjutkan sesi RSVD sebelumnya...")
    print(f"[INFO] {len(completed_indices)} kombinasi sudah dievaluasi.")
    print(f"[INFO] MSE terbaik saat ini: {best_rsvd_error:.4f}")
else:
    best_rsvd_error = float('inf')
    best_rsvd_params = None
    completed_indices = []

#### 2. Looping

In [16]:
for i, params in enumerate(rsvd_combinations):
    # Lewati iterasi jika kombinasi ini sudah pernah dikerjakan
    if i in completed_indices:
        continue
    
    print(f"\n[{i+1}/{len(rsvd_combinations)}] Menguji RSVD | Params: {params}")
    
    # Inisialisasi model RSVD dengan parameter saat ini
    rsvd = RSVD(
        n_factors=params['n_factors'], 
        learning_rate=params['learning_rate'], 
        lambda_reg=params['lambda_reg'], 
        epochs=30  # Epoch kecil untuk tuning
    )
    
    # Latih RSVD menggunakan matriks laten Z
    rsvd.fit(latent_matrix_Z)
    
    # Evaluasi seberapa baik dekomposisi merekonstruksi Z (Menghitung MSE)
    # Z_rekonstruksi = U * Sigma * V^T
    reconstructed_Z = np.dot(np.dot(rsvd.U, rsvd.Sigma), rsvd.V.T)
    mse_error = np.mean(np.square(latent_matrix_Z - reconstructed_Z))
    
    print(f"   -> MSE Rekonstruksi Z: {mse_error:.4f}")
    
    # Cek apakah ini kombinasi terbaik
    if mse_error < best_rsvd_error:
        print(f"   🌟 [NEW BEST FOUND!] MSE turun ke {mse_error:.4f}. Menyimpan model...")
        best_rsvd_error = mse_error
        best_rsvd_params = params
        best_rsvd_model = rsvd

        # Menyimpan komponen RSVD jika mendapat skor terbaik
        np.save(os.path.join(hyper_parameter_folder, 'best_U.npy'), rsvd.U)
        np.save(os.path.join(hyper_parameter_folder, 'best_Sigma.npy'), rsvd.Sigma)
        np.save(os.path.join(hyper_parameter_folder, 'best_V.npy'), rsvd.V)

    completed_indices.append(i)
    with open(rsvd_progress_file, 'w') as f:
        json.dump({
            'best_error': best_rsvd_error,
            'best_params': best_rsvd_params,
            'completed_indices': completed_indices
        }, f)
        
    # Bersihkan variabel array raksasa dari memori untuk jaga-jaga
    del rsvd, reconstructed_Z
    gc.collect()

print("\n==================================================")
print(f"[HASIL TERBAIK RSVD]")
print(f"Parameter Terbaik: {best_rsvd_params}")
print(f"MSE Terendah pada Ruang Laten: {best_rsvd_error:.4f}")
print("==================================================")


[1/48] Menguji RSVD | Params: {'n_factors': 10, 'learning_rate': 0.001, 'lambda_reg': 0.001}
Epoch 001/30 | Training MSE: 0.282421
Epoch 002/30 | Training MSE: 0.282419
Epoch 003/30 | Training MSE: 0.282417
Epoch 004/30 | Training MSE: 0.282415
Epoch 005/30 | Training MSE: 0.282413
Epoch 006/30 | Training MSE: 0.282410
Epoch 007/30 | Training MSE: 0.282408
Epoch 008/30 | Training MSE: 0.282406
Epoch 009/30 | Training MSE: 0.282404
Epoch 010/30 | Training MSE: 0.282402
Epoch 011/30 | Training MSE: 0.282400
Epoch 012/30 | Training MSE: 0.282398
Epoch 013/30 | Training MSE: 0.282396
Epoch 014/30 | Training MSE: 0.282394
Epoch 015/30 | Training MSE: 0.282392
Epoch 016/30 | Training MSE: 0.282390
Epoch 017/30 | Training MSE: 0.282388
Epoch 018/30 | Training MSE: 0.282386
Epoch 019/30 | Training MSE: 0.282384
Epoch 020/30 | Training MSE: 0.282382
Epoch 021/30 | Training MSE: 0.282380
Epoch 022/30 | Training MSE: 0.282378
Epoch 023/30 | Training MSE: 0.282376
Epoch 024/30 | Training MSE: 0.2